# 00.5 sklearn Minimal Pipeline

The goal of this notebook is to build a small but complete machine-learning workflow before moving deeper into PyTorch. The workflow has the same shape as later projects: split the data, build a baseline, apply preprocessing only through the training pipeline, train a model, and evaluate it with more than one metric.

The most important lesson is data leakage. Preprocessing steps such as scaling must be fit only on the training data. A pipeline helps enforce that rule because the scaler and model are treated as one object during fitting and evaluation.

## Learning Goals

After this notebook, you should be able to:

1. Split data with `train_test_split`.
2. Explain why we split before scaling.
3. Train a basic classifier.
4. Evaluate with accuracy and a confusion matrix.
5. Understand the role of a baseline model.
6. Transfer this workflow to later neural-network tasks.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Loading Data

We use the `iris` dataset because it is small, stable, and suitable for demonstration.


In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

print("feature head / feature head:")
print(X.head())
print()
print("labelhead / target head:")
print(y.head())
print()
print("target names / target names:", iris.target_names)

## Train-Test Split

We split the data so that training and evaluation answer different questions. The training set is used to learn model parameters. The test set is held back to estimate how well the trained model behaves on data it did not learn from.

If you repeatedly tune decisions based on the test set, the test score stops being a clean estimate of generalization. Later projects will add a validation set for tuning and keep the test set for the final check.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Baseline Model

A baseline model gives you a minimum reference line before you try something more advanced.

If your complex model is not better than the baseline, the pipeline is usually the problem.


In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
dummy_pred = dummy_clf.predict(X_test)
dummy_acc = accuracy_score(y_test, dummy_pred)

print("baseline accuracy / baseline accuracy:", dummy_acc)

## Standardization and Pipeline

Many models are sensitive to feature scales. Standardization transforms each feature so that values are centered and measured on a comparable scale. The scaler itself learns statistics such as mean and standard deviation, so it must be fit only on training data.

A `Pipeline` binds preprocessing and the model together. When you call `fit`, the scaler is fit on the training split and the model is trained on the transformed training data. When you call `predict`, the same learned scaling is applied before prediction. This keeps the workflow cleaner and reduces leakage risk.

In [ ]:
pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=500)),
    ]
)

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
acc = accuracy_score(y_test, pred)

print("logistic regression accuracy / logistic regression accuracy:", acc)

Important order:

1. split first
2. fit the scaler on the training set
3. apply the same transform to the test set

This is one of the key steps for avoiding data leakage.


## Evaluation

Accuracy is useful, but it is only one summary number. A confusion matrix shows which classes are being confused with each other. A classification report gives precision, recall, and F1-score per class. Those details matter because two models can have the same accuracy while making very different kinds of mistakes.

In [ ]:
cm = confusion_matrix(y_test, pred)
report = classification_report(y_test, pred, target_names=iris.target_names)

print("confusion matrix / confusion matrix:")
print(cm)
print()
print("classification report / classification report:")
print(report)

## Comparing with the Baseline

What matters is not a raw accuracy number by itself, but how much it improves over the baseline.


In [ ]:
comparison = pd.DataFrame(
    {
        "model": ["DummyClassifier", "LogisticRegression"],
        "accuracy": [dummy_acc, acc],
    }
)

print(comparison)

In [ ]:
# Exercise 1
#
# Implement build_logreg_pipeline().
#
# The function should return a sklearn Pipeline with two named steps:
# - "scaler": StandardScaler()
# - "model": LogisticRegression(max_iter=500)
#
# The scaler must come first because the model should receive standardized
# features. The function does not need to fit the pipeline; it only constructs
# and returns it.

def build_logreg_pipeline():
    # TODO
    pass


# test_pipe = build_logreg_pipeline()
# print(test_pipe)

In [ ]:
# Exercise 1 Reference Solution

def build_logreg_pipeline_solution():
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=500)),
        ]
    )


print(build_logreg_pipeline_solution())

In [ ]:
# Exercise 2
#
# Answer in one or two full sentences:
# Why should we not fit the scaler on the full dataset before splitting into
# train and test?
#
# Your answer should mention what information the scaler learns and why learning
# it from the test set makes evaluation too optimistic.

Exercise 2 Reference Answer

Fitting the scaler on the full dataset leaks information from the test set into preprocessing, so the test score is no longer a clean estimate of performance on unseen data.

## Summary

The key lesson here is not a specific model, but the complete workflow.

You should now be able to answer:

1. Why do we split before scaling?
2. Why do we build a baseline model first?
3. What extra information does a confusion matrix provide?
4. Why is a `Pipeline` safer and cleaner?

Suggested next step:

- Phase 0 now forms a closed loop, so the next natural step is `PyTorch` tensor basics.